# COVID-19 Big Data Project (Spark Edition)
This notebook uses **Apache Spark** for large-scale data analysis and machine learning. This approach follows Big Data concepts by leveraging distributed computing for processing COVID-19 testing data.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower, count, avg
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import gradio as gr
import pandas as pd

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Covid19BigDataNotebook") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Session Initialized!")

c:\Users\Abdo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Spark Session Initialized!


In [2]:
print("Loading data...")
file_path = "corona_tested_006 (2).csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Preprocessing
df = df.filter(col("Corona").isin(["positive", "negative"]))
df = df.withColumn("label", when(col("Corona") == "positive", 1).otherwise(0))

symptom_cols = ['Cough_symptoms', 'Fever', 'Sore_throat', 'Shortness_of_breath', 'Headache']
for c in symptom_cols:
    df = df.withColumn(c, when(lower(col(c).cast("string")) == "true", 1).otherwise(0))

df = df.withColumn("Age_60_above", when(lower(col("Age_60_above").cast("string")) == "yes", 1).otherwise(0))
df = df.withColumn("Sex", when(lower(col("Sex").cast("string")) == "male", 1).otherwise(0))
df = df.withColumn("Known_contact", 
                   when(lower(col("Known_contact").cast("string")) == "contact with confirmed", 2)
                   .when(lower(col("Known_contact").cast("string")) == "abroad", 1)
                   .otherwise(0))

print("Data Loaded and Preprocessed!")
df.show(5)

Loading data...
Data Loaded and Preprocessed!
+------+----------+--------------+-----+-----------+-------------------+--------+--------+------------+---+-------------+-----+
|Ind_ID| Test_date|Cough_symptoms|Fever|Sore_throat|Shortness_of_breath|Headache|  Corona|Age_60_above|Sex|Known_contact|label|
+------+----------+--------------+-----+-----------+-------------------+--------+--------+------------+---+-------------+-----+
|     1|11-03-2020|             1|    0|          1|                  0|       0|negative|           0|  0|            1|    0|
|     2|11-03-2020|             0|    1|          0|                  0|       0|positive|           0|  0|            1|    1|
|     3|11-03-2020|             0|    1|          0|                  0|       0|positive|           0|  0|            1|    1|
|     4|11-03-2020|             1|    0|          0|                  0|       0|negative|           0|  0|            1|    0|
|     5|11-03-2020|             1|    0|          0|      

In [3]:
print("--- BIG DATA INSIGHTS ---")
total = df.count()
print(f"Total Records: {total}")

print("Result Distribution:")
df.groupBy("Corona").count().show()

print("Symptom average in Positive cases:")
df.filter(col("label") == 1).select([avg(col(c)) for c in symptom_cols]).show()

print("Age risk analysis:")
df.groupBy("Age_60_above", "Corona").count().show()

--- BIG DATA INSIGHTS ---
Total Records: 274956
Result Distribution:
+--------+------+
|  Corona| count|
+--------+------+
|positive| 14729|
|negative|260227|
+--------+------+

Symptom average in Positive cases:
+-------------------+-------------------+-------------------+------------------------+-------------------+
|avg(Cough_symptoms)|         avg(Fever)|   avg(Sore_throat)|avg(Shortness_of_breath)|      avg(Headache)|
+-------------------+-------------------+-------------------+------------------------+-------------------+
| 0.4470093013782334|0.37741869780704734|0.10360513273134632|      0.0790277683481567|0.15174146242107406|
+-------------------+-------------------+-------------------+------------------------+-------------------+

Age risk analysis:
+------------+--------+------+
|Age_60_above|  Corona| count|
+------------+--------+------+
|           1|negative| 23221|
|           1|positive|  2204|
|           0|positive| 12525|
|           0|negative|237006|
+------------+-

In [4]:
print("Training ML Model...")
feature_cols = symptom_cols + ['Age_60_above', 'Sex', 'Known_contact']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df).select("features", "label")

train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

# Random Forest is a strong ensemble algorithm for distributed Spark environments
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)
model = rf.fit(train_data)

predictions = model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Model Test Accuracy: {accuracy * 100:.2f}%")

Training ML Model...
Model Test Accuracy: 96.73%


In [5]:
def predict_corona(cough, fever, sore_throat, shortness_of_breath, headache, age_60, sex, contact):
    input_dict = {
        'Cough_symptoms': 1 if cough else 0,
        'Fever': 1 if fever else 0,
        'Sore_throat': 1 if sore_throat else 0,
        'Shortness_of_breath': 1 if shortness_of_breath else 0,
        'Headache': 1 if headache else 0,
        'Age_60_above': 1 if age_60 else 0,
        'Sex': 1 if sex == 'Male' else 0,
        'Known_contact': 2 if contact == 'Contact with confirmed' else (1 if contact == 'Abroad' else 0)
    }
    
    input_df = spark.createDataFrame([input_dict])
    input_data = assembler.transform(input_df)
    result = model.transform(input_data).collect()[0]
    
    prob = result['probability'][1] * 100
    if result['prediction'] == 1.0:
        return f"High Likelihood ({prob:.1f}%)"
    return f"Low Likelihood ({prob:.1f}%)"

interface = gr.Interface(
    fn=predict_corona, 
    inputs=["checkbox", "checkbox", "checkbox", "checkbox", "checkbox", "checkbox", 
            gr.Radio(["Male", "Female"]), 
            gr.Dropdown(["Other", "Abroad", "Contact with confirmed"])],
    outputs="text",
    title="Spark COVID-19 Predictor"
)
interface.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
